# Sentiment Analysis Model Comparison

## Project Overview
This project focuses on sentiment analysis of product reviews. We classify reviews into negative, neutral, and positive categories and compare multiple models.
The overall task is to predict whether each review is:
- `0` = negative
- `1` = neutral
- `2` = positive

## 1. Imports

The first step and most important step is to imports the main libraries used in the notebook.

Key libraries:
- `pandas` / `numpy`: data handling
- `torch`: deep learning backend
- `sklearn`: train/test split and evaluation metrics
- `transformers`: Hugging Face BERT model and tokenizer
- `datasets`: converts data into Hugging Face dataset format

In [125]:
!pip install tensorflow

In [126]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
import numpy as np

## Install / update required packages

This installs or updates the Hugging Face training packages.

In [127]:
import sys
!{sys.executable} -m pip install -U "transformers[torch]" "accelerate>=1.1.0"

## Google Drive setup 

This code is commented out. They were probably used when running the notebook in Google Colab.

If you run locally, you can ignore these cells as long as `AllProductReviews.csv` is in the same folder as the notebook.


In [85]:
# from google.colab import drive
# drive.mount('/content/drive')

In [86]:
# cd /content/drive/MyDrive/NLP/Assignment3/dataset

## Load and clean the dataset

In this step we read the CSV file, renames the useful columns, keeps only the review text and rating, then removes missing values.

Original columns used:
- `ReviewBody` → renamed to `reviewText`
- `ReviewStar` → renamed to `overall`

At this stage, the dataset still has star ratings, not sentiment labels yet.

In [128]:
df = pd.read_csv("dataset/AllProductReviews.csv")
print(f"Raw Dataset: \n{df.head(1)}")

df = df.rename(columns={
    "ReviewBody": "reviewText",
    "ReviewStar": "overall"
})

# Keep only the needed column
df = df[["reviewText", "overall"]]

# Remove the missing values (if exists)
print("")
print(df[["reviewText", "overall"]].isnull().sum())
df = df.dropna(subset=["reviewText", "overall"])

print("")
print(f"Dataset: \n{df.head(1)}")

Raw Dataset: 
                               ReviewTitle  \
0  Honest review of an edm music lover\r\n   

                                          ReviewBody  ReviewStar  \
0  No doubt it has a great bass and to a great ex...           3   

            Product  
0  boAt Rockerz 255  

reviewText    0
overall       0
dtype: int64

Dataset: 
                                          reviewText  overall
0  No doubt it has a great bass and to a great ex...        3


## Convert star ratings into sentiment labels

This step converts the original 1–5 star ratings into 3 sentiment classes:

- 1 or 2 stars → `0` = negative
- 3 stars → `1` = neutral
- 4 or 5 stars → `2` = positive

This makes the task a **3-class classification problem**.


In [129]:
def convert_rating_to_label(rating):  # Convert the rating to Negative, Neutral, and Positive
    if rating in [1, 2]:
        return 0  # Negative
    elif rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

## Apply the sentiment label conversion

This creates a new column called `label`, which is what the model will learn to predict.

In [130]:
df["label"] = df["overall"].apply(convert_rating_to_label)
print(df.head(2))

                                          reviewText  overall  label
0  No doubt it has a great bass and to a great ex...        3      1
1  This  earphones are unreliable, i bought it be...        1      0


## Check class distribution

This counts how many negative, neutral, and positive examples are in the dataset.

This is important because if one class is much larger than the others, the model may become biased toward that class.

In [131]:
label_names = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

label_counts = df["label"].value_counts().sort_index()
label_counts_named = label_counts.rename(index=label_names)

print(label_counts_named)

label
negative    3432
neutral     1503
positive    9402
Name: count, dtype: int64


## Split data into train, validation, and test sets

This creates three datasets:

- **Training set**: used to train the model
- **Validation set**: used to tune/check the model during development
- **Test set**: used at the end to measure final performance

The split is stratified, meaning each split should keep roughly the same proportion of negative, neutral, and positive reviews. This is is also kept the same across all the methods being tested to ensure we can evaluate methods fairly 

In [132]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df["reviewText"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("X_train size:", len(X_train))
print("X_test size:", len(X_test))
print("X_val size:", len(X_val))

X_train size: 11469
X_test size: 1434
X_val size: 1434


**These steps are repeated for each model to reset everything so each one starts fresh.**

## Method 1: BERT
### Convert pandas data into Hugging Face Dataset format

BERT training with `Trainer` expects data in Hugging Face `Dataset` format.

This code converts the train, validation, and test splits into that format.

In [107]:
# Converts a pandas into a normal Python list
train_set = Dataset.from_dict(
    {
        "reviewText": X_train.to_list(),
        "label": y_train.to_list()
    }
)

test_set = Dataset.from_dict(
    {
        "reviewText": X_test.to_list(),
        "label": y_test.to_list()
    }
)

val_set = Dataset.from_dict(
    {
        "reviewText": X_val.to_list(),
        "label": y_val.to_list()
    }
)

### Load BERT tokenizer

This uses `bert-base-uncased`, which means:
- BERT is pretrained
- text is lowercased
- words are split into tokens/subword tokens

The tokenizer converts review text into numbers that BERT can understand.

In [108]:
model = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model)


def tokenizer_function(text):
    return tokenizer(
        text["reviewText"],
        padding=True,
        truncation=True,
        max_length=128
    )

### Tokenise all datasets

This applies the BERT tokenizer to the train, validation, and test sets.

Each review becomes:
- `input_ids`: token IDs
- `attention_mask`: tells BERT which tokens are real and which are padding
- `label`: the target class

In [109]:
tokenized_train_set = train_set.map(tokenizer_function, batched=True)
tokenized_test_set = test_set.map(tokenizer_function, batched=True)
tokenized_val_set = val_set.map(tokenizer_function, batched=True)

print(tokenized_train_set[0])

Map: 100%|██████████| 1434/1434 [00:00<00:00, 20576.07 examples/s]

{'reviewText': 'Pros -Solid builtGood sound qualityFits well in the earBetter calling qualityCons -You get Static shocks sometimes\r\n', 'label': 2, 'input_ids': [101, 4013, 2015, 1011, 5024, 2328, 24146, 2614, 3737, 8873, 3215, 2092, 1999, 1996, 4540, 20915, 3334, 4214, 3737, 8663, 2015, 1011, 2017, 2131, 10763, 28215, 2823, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

### Set dataset format for PyTorch

This tells the Hugging Face datasets to return PyTorch tensors for the columns needed by BERT.

In [110]:
tokenized_train_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

tokenized_test_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

tokenized_val_set.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

### Explore the tokenizer
These next cells are not part of training. They are just to help understand what BERT tokenization is doing.

In [111]:
sample_text = train_set["reviewText"][0]
sample_tokens = tokenizer(sample_text, padding=True, truncation=True, max_length=128)

print("Original text (first 200 chars):\n", sample_text[:200])
print("\nNumber of input_ids:", len(sample_tokens["input_ids"]))
print("First 20 token IDs:", sample_tokens["input_ids"][:20])
print("First 20 decoded tokens:", tokenizer.convert_ids_to_tokens(sample_tokens["input_ids"][:20]))
print("Attention mask (first 30 values):", sample_tokens["attention_mask"][:30])

Original text (first 200 chars):
 Pros -Solid builtGood sound qualityFits well in the earBetter calling qualityCons -You get Static shocks sometimes


Number of input_ids: 28
First 20 token IDs: [101, 4013, 2015, 1011, 5024, 2328, 24146, 2614, 3737, 8873, 3215, 2092, 1999, 1996, 4540, 20915, 3334, 4214, 3737, 8663]
First 20 decoded tokens: ['[CLS]', 'pro', '##s', '-', 'solid', 'built', '##good', 'sound', 'quality', '##fi', '##ts', 'well', 'in', 'the', 'ear', '##bet', '##ter', 'calling', 'quality', '##con']
Attention mask (first 30 values): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


### Observe subword tokenization

In [112]:
test_words = ["Worked", "Gadget", "Disappoints"]

for word in test_words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:20s} → {tokens}")

Worked               → ['worked']
Gadget               → ['ga', '##dget']
Disappoints          → ['di', '##sa', '##pp', '##oint', '##s']


### Evaluation Before fine-tuning
We evaluate the pre-trained BERT model without training it on our dataset to establish a baseline performance.

#### Create baseline BERT model and metric function

This cell creates a BERT sequence classification model with 3 output labels.

The `compute_metrics` function calculates:
- accuracy
- precision
- recall
- F1-score

These are the same types of metrics the other models should use so comparison is fair.

In [113]:
id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

baseline_model = BertForSequenceClassification.from_pretrained(
    model,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10839.40it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

#### Reusable evaluation function

This function runs validation and test evaluation for any BERT-style model.

It also appends results into the `results` list so the notebook can later create a comparison table.

In [114]:
results = []

def run_val_and_test(model, model_name, tokenized_val_set, tokenized_test_set, compute_metrics, output_dir):
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=output_dir,
            report_to="none",
            per_device_train_batch_size=16,
        ),
        compute_metrics=compute_metrics
    )

    # Validation
    val_results = trainer.evaluate(eval_dataset=tokenized_val_set)

    print(f"\nValidation metrics {model_name}")
    print("Accuracy:", val_results["eval_accuracy"])
    print("Precision:", val_results["eval_precision"])
    print("Recall:", val_results["eval_recall"])
    print("F1:", val_results["eval_f1"])

    # Test
    test_results = trainer.predict(tokenized_test_set)

    print(f"\nTest metrics {model_name}")
    print("Accuracy:", test_results.metrics["test_accuracy"])
    print("Precision:", test_results.metrics["test_precision"])
    print("Recall:", test_results.metrics["test_recall"])
    print("F1:", test_results.metrics["test_f1"])

    # Store the results
    results.append({
        "model": model_name,
        "val_accuracy": val_results["eval_accuracy"],
        "val_precision": val_results["eval_precision"],
        "val_recall": val_results["eval_recall"],
        "val_f1": val_results["eval_f1"],
        "test_accuracy": test_results.metrics["test_accuracy"],
        "test_precision": test_results.metrics["test_precision"],
        "test_recall": test_results.metrics["test_recall"],
        "test_f1": test_results.metrics["test_f1"]
    })

    return val_results, test_results


#### Run BERT before fine-tuning

This evaluates the baseline BERT model on the validation and test sets before any training happens.

In [115]:
pre_val_results, pre_test_results = run_val_and_test(
    model=baseline_model,
    model_name="before_fine_tuning",
    tokenized_val_set=tokenized_val_set,
    tokenized_test_set=tokenized_test_set,
    compute_metrics=compute_metrics,
    output_dir="/content/drive/MyDrive/NLP/Assignment3/output",
)

c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
No log,0.965401,0,0.656206,0.430607,0.656206,0.519992



Validation metrics before_fine_tuning
Accuracy: 0.6562064156206415
Precision: 0.4306068599016902
Recall: 0.6562064156206415
F1: 0.5199917786097041


c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Test metrics before_fine_tuning
Accuracy: 0.6555090655509066
Precision: 0.4296921350194227
Recall: 0.6555090655509066
F1: 0.5191057469400608


### Fine-Tuning BERT
The pre-trained BERT model is fine-tuned on the training dataset to adapt it to the sentiment classification task.

#### Create a fresh BERT model for full fine-tuning
This reloads `bert-base-uncased` as a 3-class classification model.

In [116]:
full_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13142.28it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

##### Train the fine-tuned BERT model

This is where actual BERT training happens.

Important settings:
- `num_train_epochs=3`: trains for 3 passes over the training data
- `learning_rate=2e-5`: small learning rate commonly used for BERT
- `eval_strategy=epoch`: evaluates after each epoch
- `load_best_model_at_end=True`: keeps the best validation model

In [117]:
full_trainer = Trainer(
    model=full_model,
    args=TrainingArguments(
        output_dir="/content/drive/MyDrive/NLP/Assignment3/output/bert_full_finetune",
        report_to="none",
        logging_dir="/content/drive/MyDrive/NLP/Assignment3",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        logging_steps=1000
    ),
    train_dataset=tokenized_train_set,
    eval_dataset=tokenized_val_set,
    compute_metrics=compute_metrics
)

full_trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.423894,0.838912,0.819777,0.838912,0.817464
2,0.452450,0.437865,0.850767,0.828033,0.850767,0.833219
3,0.304498,0.468914,0.847280,0.834897,0.847280,0.839893


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.55it/s]
c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.74s/it]
c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.

TrainOutput(global_step=2151, training_loss=0.3709188219227385, metrics={'train_runtime': 12903.3508, 'train_samples_per_second': 2.667, 'train_steps_per_second': 0.167, 'total_flos': 2263235840941824.0, 'train_loss': 0.3709188219227385, 'epoch': 3.0})

#### Evaluate fine-tuned BERT

This evaluates the trained BERT model on validation and test data and stores the results.

In [118]:
full_val_results, full_test_results = run_val_and_test(
    model=full_trainer.model,
    model_name="full_fine_tuning",
    tokenized_val_set=tokenized_val_set,
    tokenized_test_set=tokenized_test_set,
    compute_metrics=compute_metrics,
    output_dir="/content/drive/MyDrive/NLP/Assignment3/output/bert_full_results",
)

c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
No log,0.437939,0,0.850767,0.828033,0.850767,0.833219



Validation metrics full_fine_tuning
Accuracy: 0.8507670850767085
Precision: 0.8280331719781593
Recall: 0.8507670850767085
F1: 0.8332185026303637


c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Test metrics full_fine_tuning
Accuracy: 0.8403068340306834
Precision: 0.8098465989823658
Recall: 0.8403068340306834
F1: 0.8186117065325512


In [119]:
results_df = pd.DataFrame(results)
print(results_df)

                model  val_accuracy  val_precision  val_recall    val_f1  \
0  before_fine_tuning      0.656206       0.430607    0.656206  0.519992   
1    full_fine_tuning      0.850767       0.828033    0.850767  0.833219   

   test_accuracy  test_precision  test_recall   test_f1  
0       0.655509        0.429692     0.655509  0.519106  
1       0.840307        0.809847     0.840307  0.818612  


## Method 2: LSTM 

### LSTM tokenisation

In [ ]:
max_length = 500

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_lstm = tokenizer.texts_to_sequences(X_train)
X_val_lstm = tokenizer.texts_to_sequences(X_val)
X_test_lstm = tokenizer.texts_to_sequences(X_test)

X_train_lstm = pad_sequences(X_train_lstm, maxlen=max_length, padding="post", truncating="post")
X_val_lstm = pad_sequences(X_val_lstm, maxlen=max_length, padding="post", truncating="post")
X_test_lstm = pad_sequences(X_test_lstm, maxlen=max_length, padding="post", truncating="post")

### Make labels numpy arrays

In [ ]:
y_train_lstm = np.array(y_train)
y_val_lstm = np.array(y_val)
y_test_lstm = np.array(y_test)

### LSTM model 

In [ ]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Embedding(input_dim=10000, output_dim=32, input_length=max_length))


model.add(tf.keras.layers.Dropout(0.25))
model.add(tf.keras.layers.LSTM(units=32))
model.add(tf.keras.layers.Dropout(0.25))

model.add(tf.keras.layers.Dense(units=3, activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
batch_size = 256
model.build(input_shape=(batch_size, max_length))

model.summary()

c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (256, 500, 32)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (256, 500, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (256, 32)              │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (256, 32)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (256, 3)               │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 328,419 (1.25 MB)

 Trainable params: 328,419 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

### Training LSTM model 

In [ ]:
history = model.fit(
    X_train_lstm,
    y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    epochs=10,
    batch_size=64
)

Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 105ms/step - accuracy: 0.6557 - loss: 0.8730 - val_accuracy: 0.6562 - val_loss: 0.8556
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 103ms/step - accuracy: 0.6558 - loss: 0.8610 - val_accuracy: 0.6562 - val_loss: 0.8549
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 17s 97ms/step - accuracy: 0.6558 - loss: 0.8609 - val_accuracy: 0.6562 - val_loss: 0.8551
Epoch 4/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 17s 95ms/step - accuracy: 0.6558 - loss: 0.8601 - val_accuracy: 0.6562 - val_loss: 0.8551
Epoch 5/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 102ms/step - accuracy: 0.6558 - loss: 0.8604 - val_accuracy: 0.6562 - val_loss: 0.8567
Epoch 6/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 97ms/step - accuracy: 0.6558 - loss: 0.8602 - val_accuracy: 0.6562 - val_loss: 0.8549
Epoch 7/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 17s 93ms/step - accuracy: 0.6558 - loss: 0.8589 - val_accuracy: 0.6562 - val_loss: 0.8548
Epoch 8/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 98ms/step - accuracy: 0.6558 - loss: 0.8588

### Testing the LSTM mdel 

In [ ]:
y_pred_probs = model.predict(X_test_lstm)
y_pred = np.argmax(y_pred_probs, axis=1)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step


### Evaluating th LSTM Model 

In [139]:
from sklearn.metrics import classification_report, accuracy_score

print("Test Accuracy:", accuracy_score(y_test_lstm, y_pred))

print(classification_report(
    y_test_lstm,
    y_pred,
    target_names=["negative", "neutral", "positive"]
))

Test Accuracy: 0.6555090655509066
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       343
     neutral       0.00      0.00      0.00       151
    positive       0.66      1.00      0.79       940

    accuracy                           0.66      1434
   macro avg       0.22      0.33      0.26      1434
weighted avg       0.43      0.66      0.52      1434



c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hanaa\.vscode\nlp-group-17\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

## Results table

This converts the stored model results into a dataframe.

Right now, this table only includes the BERT methods, we will soon implement all the rest of the methods 

In [140]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd

# BERT RESULTS 
bert_preds = full_test_results.predictions.argmax(axis=1)
bert_labels = full_test_results.label_ids

bert_precision, bert_recall, bert_f1, _ = precision_recall_fscore_support(
    bert_labels, bert_preds, average='weighted', zero_division=0
)

bert_accuracy = accuracy_score(bert_labels, bert_preds)

# LSTM RESULTS 
lstm_precision, lstm_recall, lstm_f1, _ = precision_recall_fscore_support(
    y_test_lstm, y_pred, average='weighted', zero_division=0
)

lstm_accuracy = accuracy_score(y_test_lstm, y_pred)

# Creating one table 
results_table = pd.DataFrame([
    {
        "Model": "BERT",
        "Accuracy": bert_accuracy,
        "Precision": bert_precision,
        "Recall": bert_recall,
        "F1 Score": bert_f1
    },
    {
        "Model": "LSTM",
        "Accuracy": lstm_accuracy,
        "Precision": lstm_precision,
        "Recall": lstm_recall,
        "F1 Score": lstm_f1
    }
])

# Round for clean display
results_table = results_table.round(3)

results_table

,Model,Accuracy,Precision,Recall,F1 Score
0,BERT,0.840,0.81,0.840,0.819
1,LSTM,0.656,0.43,0.656,0.519
